# Feast Offline Store Visualization: Pandas and DuckDB

Feast stores feature metadata in the registry and delegates feature values to backing stores. In this project, the offline store is local Parquet and the online store is Redis. This notebook focuses on the offline Parquet data because it is the easiest place to inspect full feature history.

Install notebook dependencies from the repo root if needed:

```bash
pip install -r notebooks/requirements.txt
```

It shows the same data two ways:

- Pandas: dataframe-first exploration and plotting.
- DuckDB: SQL-first exploration directly over Parquet files.

In [ ]:
from pathlib import Path
import os

# Keep Matplotlib cache inside the repo/session-writable temp area when needed.
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

repo_root = Path.cwd()
if not (repo_root / "data" / "offline_store").exists():
    # This lets the notebook work when launched from notebooks/.
    repo_root = repo_root.parent

team_stats_path = repo_root / "data" / "offline_store" / "team_stats" / "team_stats.parquet"
matches_path = repo_root / "data" / "offline_store" / "matches" / "matches.parquet"

for path in [team_stats_path, matches_path]:
    if not path.exists():
        raise FileNotFoundError(path)

team_stats_path, matches_path

## Pandas View

Pandas loads the Parquet files into in-memory dataframes. This is usually the fastest path for quick inspection, notebook charts, and ad hoc transformations.

In [ ]:
team_stats = pd.read_parquet(team_stats_path).sort_values(["event_timestamp", "team_name"])
matches = pd.read_parquet(matches_path).sort_values("event_timestamp")

print("team_stats", team_stats.shape)
display(team_stats.head())

print("matches", matches.shape)
display(matches.head())

In [ ]:
display(team_stats.dtypes.to_frame("dtype"))
display(matches.dtypes.to_frame("dtype"))

In [ ]:
latest_team_stats = (
    team_stats.sort_values("event_timestamp")
    .groupby("team_name", as_index=False)
    .tail(1)
    .sort_values(["wins", "goals_for", "goals_against"], ascending=[False, False, True])
    .reset_index(drop=True)
)

display(latest_team_stats.head(20))

In [ ]:
top_teams = latest_team_stats.head(12).copy()

fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(data=top_teams, y="team_name", x="wins", ax=ax, color="#4C78A8")
ax.set_title("Top teams by latest win count")
ax.set_xlabel("Wins")
ax.set_ylabel("Team")
plt.tight_layout()

In [ ]:
result_counts = matches["result"].value_counts().rename_axis("result").reset_index(name="matches")
display(result_counts)

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=result_counts, x="result", y="matches", ax=ax, palette="Set2", hue="result", legend=False)
ax.set_title("Match result distribution")
ax.set_xlabel("Result")
ax.set_ylabel("Matches")
plt.tight_layout()

In [ ]:
selected_team = "palmeiras"

team_history = team_stats.loc[team_stats["team_name"] == selected_team].sort_values("event_timestamp")
display(team_history.tail(10))

fig, ax = plt.subplots(figsize=(11, 5))
team_history.plot(x="event_timestamp", y=["wins", "draws", "losses"], ax=ax)
ax.set_title(f"Cumulative results over time: {selected_team}")
ax.set_xlabel("Event timestamp")
ax.set_ylabel("Count")
plt.tight_layout()

## DuckDB View

DuckDB lets you query the same Parquet files with SQL. It does not require loading the files into a database first.

In [ ]:
try:
    import duckdb
except ModuleNotFoundError:
    import subprocess
    import sys

    subprocess.check_call([sys.executable, "-m", "pip", "install", "duckdb"])
    import duckdb

duckdb.__version__

In [ ]:
con = duckdb.connect()

team_stats_sql_path = team_stats_path.as_posix()
matches_sql_path = matches_path.as_posix()

con.execute(f"CREATE OR REPLACE VIEW team_stats AS SELECT * FROM read_parquet('{team_stats_sql_path}')")
con.execute(f"CREATE OR REPLACE VIEW matches AS SELECT * FROM read_parquet('{matches_sql_path}')")

display(con.sql("DESCRIBE team_stats").df())
display(con.sql("DESCRIBE matches").df())

In [ ]:
display(con.sql("""
    SELECT *
    FROM team_stats
    ORDER BY event_timestamp, team_name
    LIMIT 10
""").df())

display(con.sql("""
    SELECT *
    FROM matches
    ORDER BY event_timestamp
    LIMIT 10
""").df())

In [ ]:
latest_team_stats_sql = con.sql("""
    WITH ranked AS (
        SELECT
            *,
            row_number() OVER (
                PARTITION BY team_name
                ORDER BY event_timestamp DESC
            ) AS row_num
        FROM team_stats
    )
    SELECT
        team_name,
        matches_played,
        wins,
        draws,
        losses,
        goals_for,
        goals_against,
        event_timestamp
    FROM ranked
    WHERE row_num = 1
    ORDER BY wins DESC, goals_for DESC, goals_against ASC
""").df()

display(latest_team_stats_sql.head(20))

In [ ]:
season_results = con.sql("""
    SELECT
        year(event_timestamp) AS season,
        result,
        count(*) AS matches
    FROM matches
    GROUP BY season, result
    ORDER BY season, result
""").df()

display(season_results.head(20))

season_pivot = season_results.pivot(index="season", columns="result", values="matches").fillna(0)
season_pivot.plot(kind="bar", stacked=True, figsize=(12, 6), colormap="Set2")
plt.title("Match results by season")
plt.xlabel("Season")
plt.ylabel("Matches")
plt.tight_layout()

In [ ]:
selected_team_sql = "palmeiras"

team_history_sql = con.sql(f"""
    SELECT
        event_timestamp,
        matches_played,
        wins,
        draws,
        losses,
        goals_for,
        goals_against
    FROM team_stats
    WHERE team_name = '{selected_team_sql}'
    ORDER BY event_timestamp
""").df()

display(team_history_sql.tail(10))

fig, ax = plt.subplots(figsize=(11, 5))
team_history_sql.plot(x="event_timestamp", y=["goals_for", "goals_against"], ax=ax)
ax.set_title(f"Goals for vs against over time: {selected_team_sql}")
ax.set_xlabel("Event timestamp")
ax.set_ylabel("Goals")
plt.tight_layout()

## Which One To Keep?

Use Pandas when you want notebook-native dataframe work, Python transformations, and quick charts.

Use DuckDB when you want SQL, joins, aggregations, filtering, or fast scans over Parquet without loading every row into memory first.

A practical pattern is to query with DuckDB, convert the result to a Pandas dataframe with `.df()`, then plot or model from that smaller result.